In [1]:
import pandas as pd
data = pd.read_csv("glass.csv")
df = data.copy()

In [2]:
df.shape

(214, 10)

In [3]:
df.columns

Index(['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type'], dtype='object')

In [4]:
df.head()

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


In [5]:
# Which column contains the target variable?
# The Type column holds the glass category we want to predict.

# Are all the feature columns numeric?
# Yes, every column contains numeric values.

# Is there an identifier column we should exclude?
# The dataset does not include a separate ID column, so no column needs to be dropped on that basis.

In [6]:
df["y"] = (df["Type"] == 1).astype(int)
df = df.drop("Type", axis=1)
df.head()

# Purpose of this step
# Converts the multi-class Type column into a binary label y.

# Why this is necessary
# Logistic regression in its standard form handles binary classification tasks.

,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,y
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


In [7]:
X = df.drop("y", axis=1)
y = df["y"]

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Purpose of this step
# Divides the dataset into separate training and testing subsets.

# Why this is necessary
# The model must be evaluated on data it has never seen.
# Evaluating on training data produces misleadingly optimistic results.

In [9]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Purpose of this step
# Standardizes each feature to zero mean and unit variance.

# Why this is necessary
# Logistic regression is sensitive to differences in feature magnitude.
# Standardization leads to faster and more stable convergence.

In [10]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Purpose of this step
# Implements the sigmoid activation function.

# Why this is necessary
# Logistic regression relies on sigmoid to map any real-valued score to a probability between 0 and 1.

In [11]:
def predict_probability(X, w, b):
    linear_output = X @ w + b
    prob = sigmoid(linear_output)
    return prob

In [12]:
def compute_loss(y_true, y_prob):
    log_loss = -y_true * np.log(y_prob) - (1 - y_true) * np.log(1 - y_prob)
    return np.mean(log_loss)

# Purpose of this step
# Calculates the binary cross-entropy (log loss) between true labels and predicted probabilities.

# Why this is necessary
# Log loss quantifies prediction quality; minimizing it drives the model toward better probability estimates.

In [13]:
def update_weights(X, y_true, w, b, lr):
    probs = predict_probability(X, w, b)
    residuals = probs - y_true
    w -= lr * (X.T @ residuals) / len(y_true)
    b -= lr * np.mean(residuals)
    return w, b

# Purpose of this step
# Executes a single gradient descent update to adjust the model parameters.

# Why this is necessary
# Iteratively applying gradient descent moves parameters toward values that minimize the loss.

In [14]:
w = np.zeros(X_train_scaled.shape[1])
b = 0.0
learning_rate = 0.1
num_epochs = 100

for epoch in range(num_epochs):
    w, b = update_weights(X_train_scaled, y_train.values, w, b, learning_rate)

# Purpose of this step
# Runs the full training loop, iterating over the data for the specified number of epochs.

# Why this is necessary
# Repeated parameter updates allow the model to progressively minimize loss and improve predictions.

In [15]:
def predict_label(probs, threshold=0.5):
    return (probs >= threshold).astype(int)

# Purpose of this step
# Applies a decision threshold to convert continuous probabilities into discrete class labels.

# Why this is necessary
# Downstream evaluation metrics like accuracy require hard class predictions rather than probabilities.

In [16]:
p_test = predict_probability(X_test_scaled, w, b)

y_pred_05 = predict_label(p_test, threshold=0.5)
y_pred_07 = predict_label(p_test, threshold=0.7)

In [17]:
print("Accuracy (0.5):", np.mean(y_pred_05 == y_test))
print("Accuracy (0.7):", np.mean(y_pred_07 == y_test))

Accuracy (0.5): 0.8604651162790697
Accuracy (0.7): 0.7209302325581395


Unlike the perceptron, which outputs a hard YES/NO decision, logistic regression produces a probability that reflects the model's confidence in its prediction. The sigmoid function plays a central role here: it smoothly squashes the linear score into the range (0, 1), preserving gradient information near the boundary. One key limitation remains, however — logistic regression still draws a linear decision boundary, which means it cannot handle non-linearly separable problems such as XOR without additional feature engineering or a more expressive model.